# Resemble Enhance (Denoise Only)

In [ ]:
import warnings
from pathlib import Path
from tqdm import tqdm
import torch
import torchaudio
from utils.clear_memory import clear_memory

warnings.filterwarnings('ignore')

In [ ]:
input_dir_Pitt = Path('../ad_detection/data/raw/Pitt-origin')
output_dir_Pitt = Path('../ad_detection/data/denoised/Pitt-origin-Resemble')

control_files_Pitt = list((input_dir_Pitt / 'Control').glob('*.wav')) + list((input_dir_Pitt / 'Control').glob('*.mp3'))
dementia_files_Pitt = list((input_dir_Pitt / 'Dementia').glob('*.wav')) + list((input_dir_Pitt / 'Dementia').glob('*.mp3'))

input_dir_Lu = Path('../ad_detection/data/raw/Lu')
output_dir_Lu = Path('../ad_detection/data/denoised/Lu-Resemble')

control_files_Lu = list((input_dir_Lu / 'Control').glob('*.wav')) + list((input_dir_Lu / 'Control').glob('*.mp3'))
dementia_files_Lu = list((input_dir_Lu / 'Dementia').glob('*.wav')) + list((input_dir_Lu / 'Dementia').glob('*.mp3'))

## Load Model

In [ ]:
from resemble_enhance.enhancer.inference import denoise as resemble_denoise

# Device detection
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f"Device: {device}")
print("Model will be auto-downloaded on first call (from HuggingFace)")

## Denoise Function

In [ ]:
def denoise_audio(audio_path, device):
    """
    Apply Resemble Enhance for speech denoising (denoise only)
    """
    dwav, sr = torchaudio.load(str(audio_path))

    # Multi-channel to mono
    dwav = dwav.mean(0)  # (channels, time) -> (time,)

    # resemble_denoise internally resamples to 44100Hz and processes
    hwav, out_sr = resemble_denoise(dwav=dwav, sr=sr, device=device, run_dir=None)

    return hwav, out_sr

In [ ]:
def batch_denoise(files, output_subdir, device, group_name):
    """
    Batch denoising (clears memory before and after each file)

    Args:
        files: List of audio files to process
        output_subdir: Output subdirectory
        device: Compute device
        group_name: Group name (for progress display)
    """
    output_subdir.mkdir(parents=True, exist_ok=True)

    success_count = 0
    skip_count = 0
    fail_count = 0

    for audio_file in tqdm(files, desc=f"Processing {group_name}"):
        # Unify output as .wav format
        output_file = output_subdir / (audio_file.stem + '.wav')

        # Skip already processed files
        if output_file.exists():
            skip_count += 1
            continue

        try:
            clear_memory()

            # Denoise
            denoised_audio, sr = denoise_audio(audio_file, device)

            # Save (16-bit integer format)
            torchaudio.save(str(output_file), denoised_audio[None], sr)
            success_count += 1

            del denoised_audio
            clear_memory()

        except Exception as e:
            fail_count += 1
            print(f"\nFailed: {audio_file.name}: {e}")
            clear_memory()

    print(f"\n{group_name} processing complete:")
    print(f"Success: {success_count}")
    print(f"Skipped: {skip_count}")
    print(f"Failed: {fail_count}")
    print(f"Total: {len(files)}")

## Pitt Denoise

In [ ]:
clear_memory()

batch_denoise(
    dementia_files_Pitt,
    output_dir_Pitt / 'Dementia',
    device,
    group_name='Dementia'
)

clear_memory()

batch_denoise(
    control_files_Pitt,
    output_dir_Pitt / 'Control',
    device,
    group_name='Control'
)

## Lu Denoise

In [ ]:
clear_memory()

batch_denoise(
    dementia_files_Lu,
    output_dir_Lu / 'Dementia',
    device,
    group_name='Dementia'
)

clear_memory()

batch_denoise(
    control_files_Lu,
    output_dir_Lu / 'Control',
    device,
    group_name='Control'
)